In [ ]:
import os
import json

In [ ]:
POSTFIX = '_07_27a'
FILE_NAME = '_result_day_per_timept' + POSTFIX
OUT_NAME = 'compiled_<<TYPE>>_per_timestep' + POSTFIX
OUT_EXCEL = f'_result/{OUT_NAME}.xlsx'
OUT_PLOT = f'_result/rmse_<<TYPE>>{POSTFIX}.pdf'
files_data = []
for root, dirs, files in os.walk("./_result"):
    for file in files:
        if file.startswith(FILE_NAME) and file.endswith(".json"):
        # if file.startswith("_result_day_s40") and file.endswith(".json"):
            # if file.startswith("_result_day_s40") : continue
            file_path = os.path.join(root, file)
            files_data.append(json.loads(open(file_path, "r").read()))

In [ ]:
import csv
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, Border, Side

ATTR_ORDER  = ['Temp', 'RH', 'WSpd', 'WDir']
ATTR_LABEL  = {'Temp': 'Temperature', 
               'RH': 'Relative Humidity', 
               'WSpd': 'Wind Speed', 
               'WDir': 'Wind Direction'}
MODEL_ORDER = ['Persistence24', 'Climatology', 'LSTM', 'LSTM-ED', 'LSTM-FC', 'LSTM-ED-FC', 'LSTM-ED-ATTN-FC', 'LSTM-ED-CNN-FC']
MODEL_LABEL = {
    'Persistence24': 'Persistence', 
    'Climatology': 'Climatology', 
    'LSTM': 'LSTM', 
    'LSTM-ED': 'LSTM-ED', 
    'LSTM-FC': 'LSTM-FC', 
    'LSTM-ED-FC': 'LSTM-ED-FC', 
    'LSTM-ED-ATTN-FC': 'LSTM-ED-ATTN-FC', 
    'LSTM-ED-CNN-FC': 'LSTM-ED-CNN-FC'
}

# MODEL_LABEL = ['Persistence', 'Climatology', 'LSTM', 'LSTM-ED', 'LSTM-FC', 'LSTM-ED-FC', 'LSTM-ED-ATTN-FC', 'LSTM-ED-CNN-FC']
MODEL_ORDER_PT1A = ['LSTM', 'LSTM-FC', 'LSTM-ED', 'LSTM-ED-FC']
MODEL_ORDER_PT1B = ['LSTM', 'LSTM-ED', 'LSTM-FC', 'LSTM-ED-FC']
MODEL_ORDER_PT2 = ['Persistence24', 'Climatology', 'LSTM-ED-FC', 'LSTM-ED-ATTN-FC', 'LSTM-ED-CNN-FC']

# col index -> True means higher-is-better (R2), False means lower-is-better (MAE/RMSE)
METRIC_COLS = {
    3: False, 4: False, 5: True,   # train MAE, RMSE, R2
    6: False, 7: False, 8: True,   # eval  MAE, RMSE, R2
    9: False, 10: False, 11: True, # test  MAE, RMSE, R2
}



In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.transforms import blended_transform_factory

def outline_range(ws, min_col, max_col, min_row, max_row, style='medium'):
    """Draw a thick outline border around the rectangular range, preserving
    each cell's existing borders on non-boundary sides."""
    side = Side(style=style)
    for r in range(min_row, max_row + 1):
        for c in range(min_col, max_col + 1):
            cell = ws.cell(r, c)
            b = cell.border
            cell.border = Border(
                left=side if c == min_col else b.left,
                right=side if c == max_col else b.right,
                top=side if r == min_row else b.top,
                bottom=side if r == max_row else b.bottom,
            )

def outline_range_thin(ws, min_col, max_col, min_row, max_row, style='thin'):
    """Draw a thick outline border around the rectangular range, preserving
    each cell's existing borders on non-boundary sides."""
    side = Side(style=style)
    for r in range(min_row, max_row + 1):
        for c in range(min_col, max_col + 1):
            cell = ws.cell(r, c)
            cell.border = Border(side, side, side, side)

def _finalize_style(ws, num_format='0.000'):
    """Set every cell's font to Arial (preserving bold/size/italic/color) and format
    numeric data cells as numbers with 3 decimal places. Call right before wb.save()."""
    for row in ws.iter_rows():
        for cell in row:
            f = cell.font
            cell.font = Font(name='Arial', size=f.size, bold=f.bold, italic=f.italic, color=f.color)
            if isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                cell.number_format = num_format

# --- plot styling: models on the x-axis, one bar group per split ---
SPLIT_ORDER  = ['train', 'eval', 'test']
SPLIT_LABELS = {'train': 'Train', 'eval': 'Eval', 'test': 'Test'}
SPLIT_COLORS = {'train': '#2a78d6', 'eval': '#eb6834', 'test': '#1baf7a'}  # categorical slots 1-3
SURFACE, INK, INK_MUTED, GRID = '#fcfcfb', '#0b0b0b', '#898781', '#e1e0d9'
RULE         = '#c3c2b7'   # baseline / divider hairline
PLOT_NCOLS   = 2      # subplot grid: 2 columns, one attribute per cell
PLOT_ROW_H   = 4.0    # inches per subplot row
# Categorical slots in validated order - series take them in order, never cycled
# out of order (the ordering is what keeps adjacent bars colourblind-separable)
CAT_PALETTE  = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100',
                '#e87ba4', '#008300', '#4a3aa7', '#e34948']

def y_break(ax, brk, hi, bars, gap=0.10):
    """Broken y-axis: the axis still starts at 0, but 0..brk is squeezed into the
    bottom `gap` of the plot. Bars stay continuous from 0; a '...' drawn inside
    each one marks where the axis skips."""
    def fwd(y):
        y = np.asarray(y, dtype=float)
        return np.where(y <= brk, gap * y / brk,
                        gap + (1 - gap) * (y - brk) / (hi - brk))

    def inv(d):
        d = np.asarray(d, dtype=float)
        return np.where(d <= gap, d * brk / gap,
                        brk + (d - gap) * (hi - brk) / (1 - gap))

    ax.set_yscale('function', functions=(fwd, inv))
    ax.set_ylim(0, hi)
    # 0 plus ticks in the resolved range only - nothing inside the squeezed band
    ax.set_yticks([0] + [t for t in MaxNLocator(nbins=5).tick_values(brk, hi)
                         if brk <= t <= hi])
    # mark the skip inside each (unbroken) bar, over the squeezed band
    tr = blended_transform_factory(ax.transData, ax.transAxes)   # x: data, y: axes
    for rect in bars:
        ax.text(rect.get_x() + rect.get_width() , gap, '...', transform=tr,
                rotation=90, ha='center', va='center', fontsize=9,
                color=SURFACE, zorder=6)
    # and on the axis itself, in the tick-label column
    ax.text(-0.012, gap / 2, '...', transform=ax.transAxes, rotation=90,
            ha='right', va='center', fontsize=9, color=INK_MUTED,
            clip_on=False, zorder=6)

def compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER, plot_path='_result/rmse_full.pdf', split_idx=None):
    """split_idx: draw a divider before the model at this index, splitting each
    subplot into two model groups. None means no divider."""
    acc = {}

    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if model not in MODEL_ORDER: continue
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in ('train', 'eval', 'test'):
                    full = splits.get(split, {}).get('full')
                    if not full:
                        continue
                    key = (model, attr, split)
                    if key not in acc:
                        acc[key] = {'mae': [], 'rmse': [], 'r2': [], 'count': []}
                    acc[key]['mae'].append(full['mae'])
                    acc[key]['rmse'].append(full['rmse'])
                    acc[key]['r2'].append(full['r2'])
                    acc[key]['count'].append(full['count'])

    rows = []
    for (model, attr, split), vals in sorted(acc.items()):
        n = len(vals['mae'])
        rows.append({
            'model': model, 'attr': attr, 'split': split,
            'mae':    sum(vals['mae'])   / n,
            'rmse':   sum(vals['rmse'])  / n,
            'r2':     sum(vals['r2'])    / n,
            'count':  int(sum(vals['count']) / n),
            'n_files': n,
        })

    # ----- One RMSE subplot per attribute in a 2 x 2 grid: models on x,
    # ----- train/eval/test as grouped bars. {attr -> {(model, split) -> rmse}}
    by_attr = {}
    for row in rows:
        by_attr.setdefault(row['attr'], {})[(row['model'], row['split'])] = row['rmse']

    attrs = sorted(by_attr, key=lambda a: ATTR_ORDER.index(a) if a in ATTR_ORDER else len(ATTR_ORDER))
    n_files = max((r['n_files'] for r in rows), default=0)
    n_models = max((len([m for m in MODEL_ORDER if any((m, s) in by_attr[a] for s in SPLIT_ORDER)])
                    for a in attrs), default=0)
    if not attrs or not n_models:
        print('Nothing to plot')
        return rows

    nrows  = math.ceil(len(attrs) / PLOT_NCOLS)
    fig_h  = nrows * PLOT_ROW_H
    fig, axes = plt.subplots(nrows, PLOT_NCOLS, dpi=130, squeeze=False,
                             figsize=(PLOT_NCOLS * (1.35 * n_models + 1.6), fig_h))
    fig.patch.set_facecolor(SURFACE)
    axes_list = list(axes.flat)
    width = 0.8 / len(SPLIT_ORDER)
    any_truncated = False

    for ax, attr in zip(axes_list, attrs):
        cells = by_attr[attr]
        models = [m for m in MODEL_ORDER if any((m, s) in cells for s in SPLIT_ORDER)]
        ax.set_facecolor(SURFACE)

        vmin = vmax = None
        bar_rects = []
        for j, split in enumerate(SPLIT_ORDER):
            vals = [cells.get((m, split)) for m in models]
            pos  = [i + (j - (len(SPLIT_ORDER) - 1) / 2) * width for i in range(len(models))]
            bars = ax.bar(pos, [v or 0 for v in vals], width * 0.92, zorder=3,
                          label=SPLIT_LABELS[split], color=SPLIT_COLORS[split],
                          edgecolor=SURFACE, linewidth=1.0)

            # Lowest RMSE within this split gets a bold, full-ink value label
            scored = [(v, i) for i, v in enumerate(vals) if v is not None]
            best_v, best_i = min(scored) if scored else (None, None)

            # ... and a dotted line in the split's colour, to read the other bars against
            if best_v is not None:
                ax.axhline(best_v, color=SPLIT_COLORS[split], linestyle=':',
                           linewidth=1.1, alpha=0.9, zorder=4)

            for i, (rect, v) in enumerate(zip(bars, vals)):
                if v is None:
                    continue
                vmin = v if vmin is None else min(vmin, v)
                vmax = v if vmax is None else max(vmax, v)
                bar_rects.append(rect)
                ax.annotate(f'{v:.2f}', (rect.get_x() + rect.get_width() / 2, v), (0, 3),
                            textcoords='offset points', rotation=90,
                            ha='center', va='bottom', fontsize=12,
                            color=INK if i == best_i else INK_MUTED,
                            fontweight='bold' if i == best_i else 'normal')

        # Divider between the two model groups
        if split_idx is not None and 0 < split_idx < len(models):
            ax.axvline(split_idx - 0.5, color=RULE, linewidth=1.0,
                       linestyle=(0, (4, 3)), zorder=2)

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([MODEL_LABEL.get(m, m) for m in models], fontsize=8)

        # Keep 0 on the axis but skip the empty range below the bars, so the
        # differences between models stay readable
        if vmin is None:
            brk, hi, truncated = 0.0, 1.0, False
        else:
            span = (vmax - 0) or abs(vmax) * 0.1 or 1.0
            brk, hi, truncated = 0.0, vmax + 0.30 * span, False

            # span = (vmax - vmin) or abs(vmax) * 0.1 or 1.0
            # brk = vmin - 0.15 * span   # where the axis resumes after the break
            # hi  = vmax + 0.30 * span
            # truncated = brk > 0 and brk / hi > 0.12   # not worth breaking otherwise

        if truncated:
            y_break(ax, brk, hi, bar_rects)
            any_truncated = True
        else:
            ax.set_ylim(0, hi)
        ax.set_ylabel('RMSE', fontsize=9, color=INK_MUTED)
        ax.set_title(ATTR_LABEL.get(attr, attr), fontsize=10, color=INK, loc='left', pad=8)
        ax.tick_params(axis='y', labelsize=8, colors=INK_MUTED, length=0)
        ax.tick_params(axis='x', length=0)
        ax.grid(axis='y', color=GRID, linewidth=0.8, zorder=0)
        ax.set_axisbelow(True)
        for side in ('top', 'right', 'left'):
            ax.spines[side].set_visible(False)
        ax.spines['bottom'].set_color(RULE)

    # Blank out any unused grid slot (e.g. fewer than 4 attributes)
    for ax in axes_list[len(attrs):]:
        ax.set_visible(False)

    # Header band above the grid: figure title, then one shared split legend
    handles, labels = axes_list[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=len(SPLIT_ORDER), fontsize=14,
               labelcolor=INK_MUTED, loc='upper left', bbox_to_anchor=(0.004, 1 - 0.26 / fig_h))
    if any_truncated:
        fig.text(0.996, 1.0, 'y-axes broken: the empty range below the bars is skipped',
                 fontsize=8, color=INK_MUTED, ha='right', va='top')
    fig.suptitle('RMSE by model' + (f'  (mean of {n_files} runs)' if n_files > 1 else ''),
                 fontsize=16, color=INK, x=0.004, y=1.0, ha='left', va='top')
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.5 / fig_h))

    fig.savefig(plot_path, bbox_inches='tight', facecolor=SURFACE, format='pdf')
    fig.savefig(plot_path.replace('.pdf', '.png'), bbox_inches='tight', facecolor=SURFACE, format='png')
    plt.show()
    print(f'Written plot to {plot_path}')

    return rows

rows = compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER_PT1A, plot_path=OUT_PLOT.replace('<<TYPE>>', 'pt1a'), split_idx=2)
# rows = compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER_PT1B, plot_path=OUT_PLOT.replace('<<TYPE>>', 'pt1b'))
rows = compile_full_results(files_data, MODEL_ORDER=MODEL_ORDER_PT2, plot_path=OUT_PLOT.replace('<<TYPE>>', 'pt2'), split_idx=None)
# for r in rows:
#     print(r)

In [ ]:
# Per-day RMSE plots: one pooled RMSE per (model, attr, forecast-day), pooling the
# eval/test splits (and all result files) by summed squared error (rmse**2 * count),
# then re-taking the root -> the RMSE over the union of those samples.
# Same 2 x 2 attribute grid as compile_full_results, but each subplot groups by
# forecast day, with one bar per model inside the day group.
# Requires: the plot styling constants + ATTR_ORDER / ATTR_LABEL / MODEL_LABEL /
# MODEL_COLORS from the cells above, and files_data (loader cell).

DAY_SPLITS = ['eval', 'test']   # splits pooled into the per-day RMSE

def compile_day_results(files_data, MODEL_ORDER=MODEL_ORDER, plot_path='_result/rmse_day.png'):
    # (model, attr, day) -> pooled summed-squared-error + count across the pooled splits/files
    acc = {}
    days_seen = set()
    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if model not in MODEL_ORDER: continue
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in DAY_SPLITS:
                    full = splits.get(split, {}).get('full')
                    if not full:
                        continue
                    for day, dm in (full.get('days') or {}).items():
                        if not dm:
                            continue
                        rmse, count = dm.get('rmse'), dm.get('count')
                        if rmse is None or count is None:
                            continue
                        d = int(day)
                        days_seen.add(d)
                        a = acc.setdefault((model, attr, d), {'sum_se': 0.0, 'count': 0})
                        a['sum_se'] += (rmse ** 2) * count   # rmse^2 * n = summed squared error
                        a['count']  += count

    day_list = sorted(days_seen)

    # {attr -> {(model, day) -> pooled RMSE}} for plotting, plus the returned row shape
    by_attr, pivot = {}, {}
    for (model, attr, d), a in acc.items():
        rmse = (a['sum_se'] / a['count']) ** 0.5 if a['count'] > 0 else None
        by_attr.setdefault(attr, {})[(model, d)] = rmse
        pivot.setdefault((attr, model), {'model': model, 'attr': attr, 'days': {}})['days'][d] = rmse

    def sort_key(entry):
        attr, model = entry
        ai = ATTR_ORDER.index(attr)   if attr  in ATTR_ORDER  else len(ATTR_ORDER)
        mi = MODEL_ORDER.index(model) if model in MODEL_ORDER else len(MODEL_ORDER)
        return (ai, mi)

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    attrs = sorted(by_attr, key=lambda a: ATTR_ORDER.index(a) if a in ATTR_ORDER else len(ATTR_ORDER))
    if not attrs or not day_list:
        print('Nothing to plot')
        return sorted_rows

    # Models plotted in every subplot, each taking the next categorical slot in order
    models = [m for m in MODEL_ORDER
              if any((m, d) in by_attr[a] for a in attrs for d in day_list)]
    colors = {m: CAT_PALETTE[i % len(CAT_PALETTE)] for i, m in enumerate(models)}
    width  = 0.8 / max(len(models), 1)

    nrows  = math.ceil(len(attrs) / PLOT_NCOLS)
    fig_h  = nrows * PLOT_ROW_H
    fig, axes = plt.subplots(nrows, PLOT_NCOLS, dpi=130, squeeze=False,
                             figsize=(PLOT_NCOLS * (0.95 * len(day_list) + 1.6), fig_h))
    fig.patch.set_facecolor(SURFACE)
    axes_list = list(axes.flat)

    for ax, attr in zip(axes_list, attrs):
        cells = by_attr[attr]
        ax.set_facecolor(SURFACE)

        vmax = 0.0
        per_day = {d: [] for d in day_list}   # day -> [(rmse, bar)] for the group's best label
        for j, model in enumerate(models):
            vals = [cells.get((model, d)) for d in day_list]
            pos  = [i + (j - (len(models) - 1) / 2) * width for i in range(len(day_list))]
            bars = ax.bar(pos, [v or 0 for v in vals], width * 0.92, zorder=3,
                          label=MODEL_LABEL.get(model, model),
                          color=colors[model],
                          edgecolor=SURFACE, linewidth=0.6)
            for d, rect, v in zip(day_list, bars, vals):
                if v is None:
                    continue
                vmax = max(vmax, v)
                per_day[d].append((v, rect))

        # Lowest RMSE within each day gets a bold, full-ink value label
        for d in day_list:
            if not per_day[d]:
                continue
            v, rect = min(per_day[d], key=lambda t: t[0])
            ax.annotate(f'{v:.3f}', (rect.get_x() + rect.get_width() / 2, v), (0, 3),
                        textcoords='offset points', rotation=90, ha='center', va='bottom',
                        fontsize=6.5, color=INK, fontweight='bold', zorder=4,
                        bbox=dict(boxstyle='square,pad=0.15', facecolor=SURFACE,
                                  edgecolor='none', alpha=0.85))

        ax.set_xticks(range(len(day_list)))
        ax.set_xticklabels([f'Day {d}' for d in day_list], fontsize=8)

        ax.set_ylim(0, vmax * 1.20 if vmax else 1)
        ax.set_ylabel('RMSE', fontsize=9, color=INK_MUTED)
        ax.set_title(ATTR_LABEL.get(attr, attr), fontsize=10, color=INK, loc='left', pad=8)
        ax.tick_params(axis='y', labelsize=8, colors=INK_MUTED, length=0)
        ax.tick_params(axis='x', length=0)
        ax.grid(axis='y', color=GRID, linewidth=0.8, zorder=0)
        ax.set_axisbelow(True)
        for side in ('top', 'right', 'left'):
            ax.spines[side].set_visible(False)
        ax.spines['bottom'].set_color(RULE)

    # Blank out any unused grid slot (e.g. fewer than 4 attributes)
    for ax in axes_list[len(attrs):]:
        ax.set_visible(False)

    # Header band above the grid: figure title, then one shared model legend
    handles, labels = axes_list[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=len(handles) or 1, fontsize=8,
               labelcolor=INK_MUTED, loc='upper left', bbox_to_anchor=(0.004, 1 - 0.26 / fig_h))
    fig.suptitle('RMSE by forecast day  ('
                 + ' + '.join(SPLIT_LABELS.get(s, s) for s in DAY_SPLITS) + ' pooled)',
                 fontsize=12, color=INK, x=0.004, y=1.0, ha='left', va='top')
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.5 / fig_h))

    fig.savefig(plot_path, bbox_inches='tight', facecolor=SURFACE)
    plt.show()
    print(f'Written plot to {plot_path}')

    return sorted_rows

day_rows = compile_day_results(files_data, MODEL_ORDER=MODEL_ORDER_PT2, plot_path=OUT_PLOT.replace('<<TYPE>>', 'day_pt2'))

In [ ]:
# Per-station RMSE table: one combined RMSE per (attr, station, model), pooling all
# splits (train/eval/test) and result files for each station via summed squared error
# (rmse**2 * count) -> the RMSE over the union of that station's samples. Train
# stations pool their train+eval windows; held-out test stations use their test
# windows. Stations are rows (grouped by attribute), models are columns.
# Requires: outline_range / outline_range_thin, ATTR_ORDER, MODEL_ORDER, files_data.

def compile_station_results(files_data, excel_path='_result/compiled_station_per_timestep.xlsx'):
    # (model, attr, station) -> pooled summed-squared-error + count across all splits/files
    acc = {}
    models_seen = set()
    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in ('train', 'eval', 'test'):
                    sp = splits.get(split)
                    if not isinstance(sp, dict):
                        continue
                    for station, sm in (sp.get('stations') or {}).items():
                        if not sm:
                            continue
                        rmse, count = sm.get('rmse'), sm.get('count')
                        if rmse is None or count is None:
                            continue
                        a = acc.setdefault((model, attr, station), {'sum_se': 0.0, 'count': 0})
                        a['sum_se'] += (rmse ** 2) * count
                        a['count']  += count
                        models_seen.add(model)

    # Column order = models (MODEL_ORDER first, then any extras alphabetically)
    model_cols = [m for m in MODEL_ORDER if m in models_seen] + sorted(m for m in models_seen if m not in MODEL_ORDER)

    # Pivot keyed by (attr, station): {model -> pooled RMSE}
    pivot = {}
    for (model, attr, station), a in acc.items():
        rmse = (a['sum_se'] / a['count']) ** 0.5 if a['count'] > 0 else None
        p = pivot.setdefault((attr, station), {'attr': attr, 'station': station, 'models': {}})
        p['models'][model] = rmse

    def sort_key(entry):
        attr, station = entry
        ai = ATTR_ORDER.index(attr) if attr in ATTR_ORDER else len(ATTR_ORDER)
        return (ai, station)   # group by attribute, then station id

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    # ----- CSV -----
    # with open(out_path, 'w', newline='') as f:
    #     writer = csv.writer(f)
    #     writer.writerow(['attr', 'station'] + model_cols)
    #     for row in sorted_rows:
    #         writer.writerow([row['attr'], row['station']] + [row['models'].get(m) for m in model_cols])
    # print(f'Written {len(sorted_rows)} rows to {out_path}')

    # ----- Excel -----
    wb = Workbook()
    ws = wb.active
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    bold   = Font(bold=True)

    # Header row: Attribute | Station | <model columns>
    for col, label in [(1, 'Attribute'), (2, 'Station')]:
        c = ws.cell(1, col, label); c.alignment = center; c.font = bold
    for i, m in enumerate(model_cols):
        c = ws.cell(1, 3 + i, m); c.alignment = center; c.font = bold

    DATA_START = 2
    for r_idx, row in enumerate(sorted_rows, start=DATA_START):
        ws.cell(r_idx, 1, row['attr'])
        ws.cell(r_idx, 2, row['station'])
        for i, m in enumerate(model_cols):
            ws.cell(r_idx, 3 + i, row['models'].get(m))

    # Bold the best (lowest RMSE) model in each station row
    n_models = len(model_cols)
    for r_idx in range(DATA_START, DATA_START + len(sorted_rows)):
        cands = [(ws.cell(r_idx, 3 + j).value, 3 + j) for j in range(n_models) if ws.cell(r_idx, 3 + j).value is not None]
        if not cands:
            continue
        _, best_col = min(cands)   # RMSE: lower is better
        ws.cell(r_idx, best_col).font = bold

    # Attribute row blocks (contiguous, since rows are grouped by attribute)
    attr_rows = {}
    for i, row in enumerate(sorted_rows):
        attr_rows.setdefault(row['attr'], []).append(DATA_START + i)

    # Borders: thin grid over the whole table, then medium outlines per
    # (column set x row set). Column sets: Attribute | Station | all model cols.
    last_col = 2 + n_models
    last_row = DATA_START - 1 + len(sorted_rows)
    outline_range_thin(ws, 1, last_col, 1, last_row)
    COL_SETS = [(1, 1), (2, 2), (3, last_col)]
    ROW_SETS = [(1, 1)] + [(min(rs), max(rs)) for rs in attr_rows.values()]
    for c0, c1 in COL_SETS:
        for r0, r1 in ROW_SETS:
            outline_range(ws, c0, c1, r0, r1)

    _finalize_style(ws)
    wb.save(excel_path)
    print(f'Written {len(sorted_rows)} rows to {excel_path}')
    return sorted_rows

station_rows = compile_station_results(files_data, excel_path=OUT_EXCEL.replace('<<TYPE>>', 'station'))